In [1]:
print('hello world')

hello world


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('SEP-28k-Filtered_with_Split_and_Path.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        28177 non-null  int64 
 1   Show              28177 non-null  object
 2   EpId              28177 non-null  int64 
 3   ClipId            28177 non-null  int64 
 4   SEP28k-T          28177 non-null  object
 5   SEP28k-D          28177 non-null  object
 6   Prolongation      28177 non-null  int64 
 7   Block             28177 non-null  int64 
 8   SoundRep          28177 non-null  int64 
 9   WordRep           28177 non-null  int64 
 10  Interjection      28177 non-null  int64 
 11  NoStutteredWords  28177 non-null  int64 
 12  filepath          28177 non-null  object
dtypes: int64(9), object(4)
memory usage: 2.8+ MB


In [4]:
#Split entries based on SEP28k-T SEP
T_train_df = df[df["SEP28k-T"]== 'train']
T_dev_df = df[df["SEP28k-T"]== 'dev']
T_test_df = df[df["SEP28k-T"]== 'test']

In [5]:
import torch
import torch.nn as nn
import torchaudio

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
w2v_starter = torchaudio.pipelines.WAV2VEC2_BASE

In [8]:
#wav2vec expects audio at 16khz 
w2v_starter.sample_rate

16000

In [9]:
w2v_model = w2v_starter.get_model().to(device)

In [10]:
testfilepath = T_train_df.iloc[1]['filepath']

In [11]:
print(testfilepath)

SEP-28k_CLIP/HeStutters/0/HeStutters_0_26.wav


In [12]:
import os
print(os.path.exists("SEP-28k_CLIP/HeStutters/0/HeStutters_0_26.wav"))

True


In [13]:
import soundfile as sf
#using soundfile instead of torchaudio to load because some dependencies of torchaudio doesn't seem to work with latest python ver.
raw_waveform_nd, raw_sample_rate = sf.read(testfilepath) 
print('raw sample rate:',raw_sample_rate)
#raw_waveform_nd is ndarray, so have to change to tensor when working with pytorch. 
raw_waveform = torch.from_numpy(raw_waveform_nd)

raw_waveform.shape

raw sample rate: 16000


torch.Size([48000])

In [14]:
#convert single audio file to wav2vec embedding (tensor on gpu if avaliable)
def raw_audio_to_w2v (filename):
    raw_waveform_nd, raw_sample_rate = sf.read(filename) 
    raw_waveform = torch.from_numpy(raw_waveform_nd)
    raw_waveform = raw_waveform.float() #w2v expects 32bit float
    raw_waveform = raw_waveform.unsqueeze(0) #w2v expects at least 2d tensor, first dimension is batch size, so have to unsqueeze to (1, 48000) for single file.
    raw_waveform = raw_waveform.to(device)
    embedded_output, _ = w2v_model(raw_waveform)
    return embedded_output


In [1]:
print(raw_audio_to_w2v(testfilepath))

NameError: name 'raw_audio_to_w2v' is not defined